# 01a — Auto-Caption Reference Images

Generates `.txt` sidecar caption files for each reference image using JoyCaption.
These captions are used for SDXL LoRA training in `01b_train_sdxl_lora.ipynb`.

**Runtime:** T4 is fine (captioning is light). A100 optional.

**Prerequisites:** JoyCaption is a gated model — you need a free HuggingFace account.
1. Sign up at https://huggingface.co (free)
2. Accept the model terms at https://huggingface.co/fancyfeast/llama-joycaption-beta-one-hf-llava
3. Create a read token at https://huggingface.co/settings/tokens
4. Paste it in Cell 1b below when prompted

**Alternative:** If you don't want to create an HF account, use the `llava-hf/llava-1.5-7b-hf` 
model instead (fully public, similar caption quality). Change `USE_JOYCAPTION = True` to `False` in Cell 1b.

**Flow:**
1. HuggingFace login
2. Upload reference images to Drive (or Colab tmp)
3. Run captioner on each image
4. Prepend trigger token to each caption
5. Save `.txt` sidecar files next to each image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
CHARACTER_NAME = 'Aria'       # ← change this
TRIGGER_TOKEN  = 'ohwx_aria'  # ← change this (must match what you use in training)

# Captioner choice:
# True  = JoyCaption Beta One (best quality, requires free HF account + model access)
# False = LLaVA 1.5 7B (fully public, no account needed, similar quality)
USE_JOYCAPTION = True

import os
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
CAPTION_DIR = f'{CHAR_DIR}/captions'
os.makedirs(REF_DIR, exist_ok=True)
os.makedirs(CAPTION_DIR, exist_ok=True)

print(f'Character: {CHARACTER_NAME} / trigger: {TRIGGER_TOKEN}')
print(f'Reference images dir: {REF_DIR}')
print(f'Captioner: {"JoyCaption Beta One" if USE_JOYCAPTION else "LLaVA 1.5 7B (public)"}')
print('Upload your reference images to the Drive folder above, then run the next cells.')

In [ ]:
# (Optional) Upload images directly from this notebook
from google.colab import files
import shutil

print('Select your reference images to upload...')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'{REF_DIR}/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved {fname} → {dest}')

In [ ]:
!pip install -q transformers torch Pillow huggingface_hub

from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch, os

if USE_JOYCAPTION:
    # JoyCaption Beta One — gated model. Two ways to authenticate:
    # Option 1 (recommended): Add HF_TOKEN to Colab Secrets (key icon in left sidebar → "Add secret")
    # Option 2: paste token directly below (less secure — don't commit this)
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
        print('HF_TOKEN loaded from Colab Secrets.')
    except Exception:
        # Fallback: paste token here
        from getpass import getpass
        hf_token = getpass('Paste your HuggingFace read token (hidden): ')

    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    MODEL_ID = 'fancyfeast/llama-joycaption-beta-one-hf-llava'
else:
    MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

print(f'Loading captioner: {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
).to('cuda' if torch.cuda.is_available() else 'cpu')
print('Captioner loaded.')

In [ ]:
from PIL import Image
from pathlib import Path

CAPTION_INSTRUCTION = (
    'Describe this image in detail for use as a training caption for a diffusion model. '
    'Focus on: the pose and body position, facial expression, clothing and accessories, '
    'lighting and background. Do NOT describe the character identity or face structure — '
    'that will be represented by a trigger token. Be specific and concrete. '
    'Keep the caption under 80 words.'
)

# Detect the correct image token for this model
IMAGE_TOKEN = getattr(processor, 'image_token', None) or '<image>'

def caption_image(image_path: str) -> str:
    image = Image.open(image_path).convert('RGB')

    # Try 1: multimodal list format (LLaVA 1.5 / JoyCaption alpha-two)
    # Try 2: string content with image token (JoyCaption beta-one / LLaVA-Next)
    # Try 3: plain processor call with image token in text
    prompt = None
    for fmt in ['list', 'string', 'plain']:
        try:
            if fmt == 'list':
                conversation = [{
                    'role': 'user',
                    'content': [{'type': 'image'}, {'type': 'text', 'text': CAPTION_INSTRUCTION}]
                }]
                prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            elif fmt == 'string':
                conversation = [{
                    'role': 'user',
                    'content': f'{IMAGE_TOKEN}\n{CAPTION_INSTRUCTION}'
                }]
                prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            else:
                prompt = f'USER: {IMAGE_TOKEN}\n{CAPTION_INSTRUCTION}\nASSISTANT:'
            break  # succeeded
        except Exception:
            continue

    inputs = processor(text=prompt, images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = processor.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

exts = {'.jpg', '.jpeg', '.png', '.webp'}
images = [p for p in Path(REF_DIR).iterdir() if p.suffix.lower() in exts]
print(f'Found {len(images)} reference images.')
print(f'Image token: {repr(IMAGE_TOKEN)}')

captions = {}
for img_path in sorted(images):
    print(f'Captioning {img_path.name} ...')
    try:
        raw_caption = caption_image(str(img_path))
        full_caption = f'{TRIGGER_TOKEN}, {raw_caption}'
    except Exception as e:
        print(f'  ERROR: {e} — skipping')
        continue
    captions[img_path.name] = full_caption
    txt_path = Path(REF_DIR) / (img_path.stem + '.txt')
    txt_path.write_text(full_caption)
    cap_path = Path(CAPTION_DIR) / (img_path.stem + '.txt')
    cap_path.write_text(full_caption)
    print(f'  → {full_caption[:100]}...')

print(f'\nAll {len(captions)} captions saved.')

In [ ]:
# Review and optionally edit captions before training
print('=== Caption Review ===')
for fname, caption in captions.items():
    print(f'\n[{fname}]')
    print(caption)
print('\nEdit the .txt files in Drive if any captions need adjustment before training.')

In [ ]:
# Register character in library (optional — also done in training notebook)
import sys, json
# If running from Colab, library.py isn't installed — use inline version
metadata = {
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'ref_count': len(captions),
}
meta_path = f'{CHAR_DIR}/metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json written: {meta_path}')
print('\n✅ Done. Run 01b_train_sdxl_lora.ipynb next to train the character LoRA.')